# BirdCLEF+ 2026 - 234-class multi-label classifier

Trains a unified 234-class multi-label model so that the 28 species missing from `train_audio` (3 frogs + 25 insect sonotypes) are learned directly from `train_soundscapes_labels.csv`. Replaces the single-label fastai pipeline in `birdclef_plus_2026_sound_classification_attempt_1.ipynb`.

Inputs:
- Competition data (`/kaggle/input/competitions/birdclef-2026/`).
- One or more `species-XXX-YYY` Kaggle datasets produced by `scripts/generate_spectrogram_batches.py` (single-label train_audio PNGs, parent-folder = species).
- A soundscape spectrogram Kaggle dataset produced locally by `scripts/generate_soundscape_spectrograms.py` (PNGs + `soundscape_index.csv` with multi-label rows).

Outputs: `model_multilabel_234.pkl` and `vocab.json` for the submission notebook.

## Setup

In [ ]:
! pip install -Uqq fastbook timm
import fastbook
fastbook.setup_book()

In [ ]:
from fastbook import *
from fastai.vision.all import *
import json
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

## Paths

Edit `SPECIES_FOLDERS` and `SOUNDSCAPE_DATASET_DIR` to point at the Kaggle datasets you uploaded. The competition data is mounted automatically via `kagglehub`.

In [ ]:
import kagglehub
competition_dir = Path(kagglehub.competition_download('birdclef-2026'))
print('Competition dir:', competition_dir)

SPECIES_FOLDERS = [
    Path('/kaggle/input/datasets/ucheozoemena/species-001-010'),
    Path('/kaggle/input/datasets/ucheozoemena/species-011-090'),
]
SOUNDSCAPE_DATASET_DIR = Path('/kaggle/input/datasets/ucheozoemena/soundscape-spectrograms')

for p in SPECIES_FOLDERS + [SOUNDSCAPE_DATASET_DIR]:
    print(f'{p}: exists={p.exists()}')

## Build the unified multi-label dataframe

Two sources concatenated into one dataframe with columns `image_path, labels, source, group, is_valid`:
- **train_audio**: each PNG -> one positive label = parent folder name.
- **soundscapes**: each PNG -> the semicolon-joined multi-label string from `soundscape_index.csv`.

The `group` column gives a stable id for the validation split (parent species for train_audio, `BC2026_Train_NNNN` file id for soundscapes) so windows from the same recording never straddle the split.

In [ ]:
isolated_rows = []
for folder in SPECIES_FOLDERS:
    if not folder.exists():
        print(f'Skipping missing folder: {folder}')
        continue
    for png in get_image_files(folder):
        species = png.parent.name
        isolated_rows.append({
            'image_path': str(png),
            'labels': species,
            'source': 'train_audio',
            'group': species,
        })
isolated_df = pd.DataFrame(isolated_rows)
print(f'train_audio rows: {len(isolated_df)}')
print(f'unique train_audio species: {isolated_df["labels"].nunique()}')

In [ ]:
soundscape_index = pd.read_csv(SOUNDSCAPE_DATASET_DIR / 'soundscape_index.csv')

def soundscape_group(image_name: str) -> str:
    # 'BC2026_Train_0001_S08_20250606_030007__s0.png' -> 'BC2026_Train_0001'
    parts = image_name.split('_')
    return '_'.join(parts[:3])

soundscape_df = pd.DataFrame({
    'image_path': [str(SOUNDSCAPE_DATASET_DIR / p) for p in soundscape_index['image_path']],
    'labels': soundscape_index['labels'],
    'source': 'soundscape',
    'group': soundscape_index['image_path'].map(soundscape_group),
})
print(f'soundscape rows: {len(soundscape_df)}')
print(f'unique soundscape file ids: {soundscape_df["group"].nunique()}')
soundscape_df.head()

In [ ]:
import hashlib

VAL_FRACTION = 0.2
SEED = 42

def stable_hash_in_valid(group: str, frac: float) -> bool:
    h = hashlib.md5(f'{group}:{SEED}'.encode()).digest()
    bucket = int.from_bytes(h[:4], 'big') / 2**32
    return bucket < frac

df = pd.concat([isolated_df, soundscape_df], ignore_index=True)
df['is_valid'] = df['group'].map(lambda g: stable_hash_in_valid(g, VAL_FRACTION))

print(f'total rows: {len(df)} (train={int((~df.is_valid).sum())}, valid={int(df.is_valid.sum())})')
print('source x split:')
print(df.groupby(['source', 'is_valid']).size())

## Pin vocabulary to the 234 sample-submission classes

We force the vocabulary on `MultiCategoryBlock` so the model head always emits 234 outputs in the exact column order of `sample_submission.csv`, even if some classes have zero training rows in any given run.

In [ ]:
sample_sub = pd.read_csv(competition_dir / 'sample_submission.csv')
ALL_SPECIES = [c for c in sample_sub.columns if c != 'row_id']
assert len(ALL_SPECIES) == 234, f'expected 234, got {len(ALL_SPECIES)}'

labels_seen = set()
for s in df['labels']:
    labels_seen.update(s.split(';'))
unknown = labels_seen - set(ALL_SPECIES)
assert not unknown, f'labels not in sample_submission columns: {unknown}'
missing_in_data = set(ALL_SPECIES) - labels_seen
print(f'classes in vocab but absent from training data: {len(missing_in_data)}')
if missing_in_data:
    print('  example:', sorted(missing_in_data)[:10])
    print('  (these get zero gradient signal; upload more species-XXX-YYY batches to cover them.)')

## DataBlock and dataloaders

In [ ]:
BATCH_SIZE = 64
IMG_SIZE = 224

dblock = DataBlock(
    blocks=(ImageBlock, MultiCategoryBlock(vocab=ALL_SPECIES)),
    splitter=ColSplitter('is_valid'),
    get_x=ColReader('image_path'),
    get_y=ColReader('labels', label_delim=';'),
    item_tfms=Resize(IMG_SIZE),
    batch_tfms=aug_transforms(size=IMG_SIZE, do_flip=False, max_rotate=0.0),
)
dls = dblock.dataloaders(df, bs=BATCH_SIZE)

assert list(dls.vocab) == ALL_SPECIES, 'vocab order must match sample_submission column order'
print(f'vocab size: {len(dls.vocab)}  train batches: {len(dls.train)}  valid batches: {len(dls.valid)}')
dls.show_batch(max_n=4, nrows=2)

## Class imbalance via BCE `pos_weight`

`pos_weight[c] = (N - n_c) / n_c`, clipped to `[1, 50]`. Computed only on the training split so we don't peek at validation.

In [ ]:
train_df = df[~df['is_valid']]
N = len(train_df)
pos_counts = pd.Series(0, index=ALL_SPECIES, dtype='int64')
for s in train_df['labels']:
    for lab in s.split(';'):
        pos_counts[lab] += 1

pos_counts = pos_counts.replace(0, 1)  # avoid division by zero for absent classes
raw = (N - pos_counts) / pos_counts
pos_weight = raw.clip(lower=1.0, upper=50.0).astype('float32')
pos_weight_t = torch.tensor(pos_weight.values, device=dls.device)

print('pos_weight summary:')
print(pos_weight.describe())
print('\nlowest-positive classes (most upweighted):')
print(pos_counts.sort_values().head(15))

## Learner

In [ ]:
loss_func = BCEWithLogitsLossFlat(pos_weight=pos_weight_t)
metrics = [accuracy_multi, APScoreMulti(average='macro')]

learn = vision_learner(
    dls,
    resnet50,
    loss_func=loss_func,
    metrics=metrics,
    n_out=len(ALL_SPECIES),
)
learn.summary

## Train

In [ ]:
learn.fine_tune(5)

In [ ]:
learn.fit_one_cycle(10, lr_max=1e-4)

## Export model + vocab

In [ ]:
learn.export('model_multilabel_234.pkl')
with open('vocab.json', 'w') as fp:
    json.dump(list(dls.vocab), fp)
print('Saved model_multilabel_234.pkl and vocab.json')

## Verification: per-class AP for the missing-28

The 28 species absent from `train_audio` (3 frogs + 25 insect sonotypes) are the entire reason this notebook exists. We compute their average precision on the validation split and flag any class whose AP is near zero so we know whether the soundscape supervision actually transferred.

In [ ]:
train_csv = pd.read_csv(competition_dir / 'train.csv')
taxonomy_csv = pd.read_csv(competition_dir / 'taxonomy.csv')
train_audio_species = set(train_csv['primary_label'].unique())
missing28 = sorted(set(taxonomy_csv['primary_label']) - train_audio_species)
assert len(missing28) == 28, f'expected 28 missing, got {len(missing28)}'
print('Missing-28 classes:', missing28)

In [ ]:
preds, targs = learn.get_preds(dl=dls.valid)
preds_np = preds.cpu().numpy()
targs_np = targs.cpu().numpy()
vocab_index = {name: i for i, name in enumerate(dls.vocab)}

report_rows = []
for sp in missing28:
    j = vocab_index[sp]
    n_pos = int(targs_np[:, j].sum())
    if n_pos == 0:
        ap = float('nan')
    else:
        ap = float(average_precision_score(targs_np[:, j], preds_np[:, j]))
    report_rows.append({'species': sp, 'val_positives': n_pos, 'ap': ap})

report = pd.DataFrame(report_rows).sort_values('ap', na_position='last')
print(report.to_string(index=False))

low_ap = report[(report['val_positives'] > 0) & (report['ap'] < 0.05)]
no_val = report[report['val_positives'] == 0]
print(f'\nMissing-28 with AP < 0.05 (likely collapsed): {len(low_ap)}')
if len(low_ap):
    print(low_ap['species'].tolist())
print(f'Missing-28 with no validation positives (split too small for them): {len(no_val)}')
if len(no_val):
    print(no_val['species'].tolist())

In [ ]:
# Macro-AP across all 234 classes (only those with >=1 validation positive count)
aps = []
for j in range(targs_np.shape[1]):
    if targs_np[:, j].sum() == 0:
        continue
    aps.append(average_precision_score(targs_np[:, j], preds_np[:, j]))
print(f'Macro-AP over {len(aps)} classes with validation positives: {np.mean(aps):.4f}')